# Torrents To Google Drive Downloader

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install libtorrent

In [ ]:
!pip install -q libtorrent

## 3. Settings
Paste a magnet link, .torrent URL, or .torrent file path.

In [ ]:
#@title Settings
link = ""  #@param {type:"string"}
drive_folder = "/content/drive/MyDrive/Torrent"  #@param {type:"string"}
use_local_buffer = False  #@param {type:"boolean"}
metadata_timeout_min = 5  #@param {type:"integer"}

# link: magnet link, .torrent URL, or path to a .torrent file
# use_local_buffer: download to Colab's local disk first, then copy to Drive (faster).
#   Automatically falls back to direct-to-Drive if the torrent won't fit locally.

## 4. Start download

In [ ]:
import libtorrent as lt
import os, time, shutil, tempfile, urllib.request
from IPython.display import clear_output

LOCAL_DIR = "/content/torrent_tmp"
os.makedirs(LOCAL_DIR, exist_ok=True)
if drive_folder.startswith("/content/drive/MyDrive") and not os.path.isdir("/content/drive/MyDrive"):
    raise RuntimeError("Google Drive isn't mounted - run the mount cell first, or files would go to "
                       "temporary local storage and be lost when the runtime resets.")
os.makedirs(drive_folder, exist_ok=True)

def hsize(n):
    for u in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024:
            return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"

def heta(sec):
    if sec is None:
        return "--"
    sec = int(sec)
    h, r = divmod(sec, 3600)
    m, s = divmod(r, 60)
    return f"{h}h {m:02d}m" if h else f"{m}m {s:02d}s"

link = link.strip()
if not link:
    raise ValueError("Paste a magnet link or .torrent URL/path in the settings cell first.")

ses = lt.session({
    "user_agent": "libtorrent/2.0",
    "alert_mask": lt.alert.category_t.storage_notification | lt.alert.category_t.error_notification,
    "dht_bootstrap_nodes": "router.bittorrent.com:6881,dht.transmissionbt.com:6881,dht.libtorrent.org:25401",
})

# Accept magnet links, .torrent URLs, or local .torrent paths
if link.startswith("magnet:"):
    params = lt.parse_magnet_uri(link)
else:
    path = link
    if link.startswith(("http://", "https://")):
        path = os.path.join(tempfile.gettempdir(), "download.torrent")
        req = urllib.request.Request(link, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            if resp.status != 200:
                raise RuntimeError(f"HTTP {resp.status} while fetching the .torrent file")
            with open(path, "wb") as fh:
                shutil.copyfileobj(resp, fh)
    params = lt.add_torrent_params()
    try:
        params.ti = lt.torrent_info(path)
    except Exception as e:
        raise ValueError(f"Not a valid .torrent file ({e})")

params.save_path = LOCAL_DIR if use_local_buffer else drive_folder
handle = ses.add_torrent(params)

def stop():
    try:
        ses.remove_torrent(handle)
    except Exception:
        pass

def partial_note():
    where = LOCAL_DIR if buffered else drive_folder
    print(f"Partial data may remain in {where} (local data is lost if the Colab runtime resets). "
          "Re-running with the same link may resume it.")

buffered = use_local_buffer
notes = []  # messages re-printed on every progress refresh so clear_output doesn't wipe them

def note(msg):
    notes.append(msg)
    print(msg)

try:
    # 1) Metadata (with timeout)
    print("Fetching metadata...")
    t0 = time.time()
    while not handle.status().has_metadata:
        if time.time() - t0 > metadata_timeout_min * 60:
            raise TimeoutError(f"No metadata after {metadata_timeout_min} min - dead magnet or no peers.")
        time.sleep(1)

    ti = handle.torrent_file()
    name, total = ti.name(), ti.total_size()
    print(f"Metadata acquired: {name} ({hsize(total)})")

    # Drive space check (advisory only: the mount's reported free space may not match your real quota)
    try:
        drive_free = shutil.disk_usage(drive_folder).free
        if drive_free < total:
            note(f"⚠️ Drive reports only {hsize(drive_free)} free for a {hsize(total)} torrent. "
                 "Interrupt now if that's a problem.")
    except OSError:
        pass

    # 2) Decide where to download: local buffer if it fits, else straight to Drive
    buffered = use_local_buffer
    if buffered:
        free = shutil.disk_usage(LOCAL_DIR).free
        required = total + max(1 << 30, int(total * 0.02))  # 1 GiB or 2% headroom, whichever is larger
        if required > free:
            note(f"Not enough local disk ({hsize(free)} free, ~{hsize(required)} needed) - "
                 "switching to direct-to-Drive (slower).")
            handle.move_storage(drive_folder)
            t1 = time.time()
            moved = False
            while not moved:
                if time.time() - t1 > 120:
                    raise TimeoutError("Timed out moving storage to Drive.")
                err = handle.status().errc
                if err.value():
                    raise RuntimeError(err.message())
                for a in ses.pop_alerts():
                    if isinstance(a, lt.storage_moved_failed_alert):
                        raise RuntimeError("Could not switch to Drive: " + a.message())
                    if isinstance(a, lt.storage_moved_alert):
                        moved = True
                time.sleep(0.2)
            buffered = False

    # 3) Download loop
    start = time.time()
    while True:
        s = handle.status()
        if s.errc.value():
            raise RuntimeError(s.errc.message())

        remaining = s.total_wanted - s.total_wanted_done
        eta = remaining / s.download_rate if s.download_rate > 0 else None

        clear_output(wait=True)
        for n in notes:
            print(n)
        print(name)
        print(f"{s.progress * 100:.2f}% | {s.state.name} | {hsize(s.total_wanted_done)} / {hsize(s.total_wanted)}")
        print(f"down {s.download_rate / 1e6:.2f} MB/s | up {s.upload_rate / 1e6:.2f} MB/s | "
              f"seeds {s.num_seeds} | peers {s.num_peers} | ETA {heta(eta)}")

        if s.is_finished:
            break
        time.sleep(3)

    elapsed = int(time.time() - start)
    stop()  # stop seeding
    print(f"\nDownload complete in {elapsed // 60}m {elapsed % 60}s.")

    # 4) Copy from local buffer to Drive
    if buffered:
        print("Copying to Drive...")
        src = os.path.join(LOCAL_DIR, name)
        dst = os.path.join(drive_folder, name)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            shutil.rmtree(src)
        else:
            shutil.copy2(src, dst)
            os.remove(src)
        print("Saved to:", dst)
    else:
        print("Saved to:", os.path.join(drive_folder, name))

except KeyboardInterrupt:
    stop()
    print("Stopped by user.")
    partial_note()
except Exception as e:
    stop()
    print("❌ Error:", e)
    partial_note()